<a href="https://colab.research.google.com/github/MBR4V0/Python/blob/main/chamada_qrcode_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chamada por QR Code — Balé, Judô, Futebol, Teatro e Artes

Aplicação web (Flask + SQLite + HTML5/Tailwind/JavaScript) rodando no Colab:

1. **Professor** abre `/prof`, entra com o PIN, escolhe a turma e vê um **QR Code** (ele se renova sozinho a cada 30 s).
2. **Aluno** aponta a câmera do celular para o QR, vê a lista **da turma**, toca no próprio nome e confirma.
3. O painel do **professor** atualiza em tempo real; ele **confere** (pode corrigir qualquer nome) e **finaliza**.
4. A chamada fica gravada e pode ser baixada em CSV (abre direto no Excel/Power BI).

Os alunos são **fictícios**: o sexo e a idade são sorteados a partir dos inscritos ativos da sua planilha, e os nomes vêm de listas genéricas
(nomes que coincidam com algum nome da planilha são descartados).

**Ordem:** execute as células de cima para baixo. A célula 5 imprime o link que o professor deve abrir.

## 1. Dependências

In [1]:
!pip -q install flask

## 2. Gerador de alunos fictícios

In [2]:
%%writefile gerar_alunos.py
"""Gera alunos FICTÍCIOS por turma, usando a planilha apenas como base estatística.

- O perfil (sexo e idade) é sorteado entre inscritos ATIVOS/PRESENCIAL da planilha.
- Os nomes são sorteados de listas genéricas. Nenhum nome, CPF ou endereço real é lido
  para dentro do resultado; nomes que coincidam com algum NOME ABREVIADO da planilha são descartados.
"""
import random
import re
import sys
import unicodedata

import pandas as pd

# turma -> proporção de meninos (ajuste à vontade)
TURMAS = {"Balé": 0.10, "Judô": 0.65, "Futebol": 0.85, "Teatro": 0.45, "Artes": 0.50}

FEM = ["Ana", "Beatriz", "Camila", "Daniela", "Eduarda", "Fernanda", "Gabriela", "Helena", "Isabela",
       "Julia", "Larissa", "Manuela", "Natália", "Olívia", "Paula", "Rafaela", "Sofia", "Tainá",
       "Valentina", "Yasmin", "Alice", "Bianca", "Clara", "Débora", "Estela", "Flávia", "Heloísa",
       "Laura", "Marina", "Nicole", "Pietra", "Renata", "Sabrina", "Vitória", "Luana", "Melissa"]
MASC = ["Arthur", "Bruno", "Caio", "Davi", "Enzo", "Felipe", "Gabriel", "Heitor", "Igor", "João",
        "Kauã", "Lucas", "Mateus", "Nicolas", "Otávio", "Pedro", "Rafael", "Samuel", "Thiago", "Vitor",
        "Yuri", "Anderson", "Benício", "Cauê", "Diego", "Eduardo", "Fábio", "Henrique", "Joaquim",
        "Leonardo", "Miguel", "Noah", "Ricardo", "Tomás", "Wesley", "Murilo"]
SOBRE = ["Almeida", "Barbosa", "Cardoso", "Dias", "Esteves", "Ferreira", "Gomes", "Henrique", "Lima",
         "Machado", "Nogueira", "Oliveira", "Pereira", "Queiroz", "Ribeiro", "Santos", "Teixeira",
         "Uchoa", "Vieira", "Xavier", "Moreira", "Campos", "Rocha", "Freitas", "Araújo", "Batista"]


def _norm(txt):
    txt = unicodedata.normalize("NFKD", str(txt)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z]", "", txt.lower())


def carregar_pool(planilha, aba="Inscritos 2024"):
    bruto = pd.read_excel(planilha, sheet_name=aba, header=None)
    lin = bruto.index[bruto.iloc[:, 0].astype(str).str.strip().eq("GERAL")][0]
    df = pd.read_excel(planilha, sheet_name=aba, header=lin)
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    reais = {_norm(x) for x in df["NOME ABREVIADO"].dropna()}
    ativos = df[df["SITUAÇÃO"].astype(str).str.upper().isin(["ATIVO", "PRESENCIAL"])]
    pool = pd.DataFrame({
        "sexo": ativos["SEXO"].astype(str).str.strip().str.title(),
        "idade": pd.to_numeric(ativos["IDADE"], errors="coerce"),
    }).dropna()
    pool = pool[(pool["idade"] >= 5) & (pool["idade"] <= 17)]
    return pool, reais


def gerar(planilha, saida="alunos_ficticios.csv", por_turma=24, seed=7):
    rng = random.Random(seed)
    pool, reais = carregar_pool(planilha)
    idades = {s: pool.loc[pool["sexo"] == s, "idade"].astype(int).tolist() for s in ("Masculino", "Feminino")}
    usados, linhas = set(), []
    for turma, p_masc in TURMAS.items():
        for _ in range(por_turma):
            sexo = "Masculino" if rng.random() < p_masc else "Feminino"
            idade = rng.choice(idades[sexo])
            while True:
                nome = "%s %s. %s." % (rng.choice(MASC if sexo == "Masculino" else FEM),
                                       rng.choice(SOBRE)[0], rng.choice(SOBRE)[0])
                chave = _norm(nome)
                if chave not in usados and chave not in reais:
                    usados.add(chave)
                    break
            linhas.append({"turma": turma, "nome": nome, "sexo": sexo, "idade": idade})
    out = pd.DataFrame(linhas).sort_values(["turma", "nome"]).reset_index(drop=True)
    out.to_csv(saida, index=False, encoding="utf-8")
    return out


if __name__ == "__main__":
    arq = sys.argv[1] if len(sys.argv) > 1 else "Inscritos_-_Andamento_v2.xlsx"
    res = gerar(arq)
    print(res.groupby("turma").agg(alunos=("nome", "size"), idade_media=("idade", "mean"),
                                   meninos=("sexo", lambda s: (s == "Masculino").sum())).round(1))

Writing gerar_alunos.py


## 3. Aplicação (servidor)

In [3]:
%%writefile app_chamada.py
"""Chamada por QR Code — Flask + SQLite.

Fluxo:
  1. Professor entra em /prof (PIN), escolhe a turma e abre a chamada -> aparece um QR Code que se renova.
  2. Aluno aponta a câmera para o QR, abre o link no celular, escolhe o próprio nome e confirma.
  3. O painel do professor atualiza sozinho; o professor confere (pode corrigir) e finaliza.

Variáveis de ambiente: CHAMADA_PIN (padrão 1234 — TROQUE), CHAMADA_SECRET, CHAMADA_DB,
CHAMADA_ROTACAO (segundos do QR, padrão 30), CHAMADA_TZ.
"""
import csv
import hashlib
import hmac
import io
import os
import secrets
import sqlite3
import threading
import time
from datetime import datetime, timedelta, timezone
from functools import wraps

from flask import (Flask, Response, jsonify, redirect, render_template_string,
                   request, session)

DB = os.environ.get("CHAMADA_DB", "chamada.db")
PIN = os.environ.get("CHAMADA_PIN", "1234")
SECRET = os.environ.get("CHAMADA_SECRET") or secrets.token_hex(16)
ROTACAO = int(os.environ.get("CHAMADA_ROTACAO", "30"))
TURMAS = ["Balé", "Judô", "Futebol", "Teatro", "Artes"]
ALUNOS_CSV = os.environ.get("CHAMADA_ALUNOS", "alunos_ficticios.csv")

try:
    from zoneinfo import ZoneInfo
    TZ = ZoneInfo(os.environ.get("CHAMADA_TZ", "America/Sao_Paulo"))
except Exception:  # sem tzdata: UTC-3 fixo
    TZ = timezone(timedelta(hours=-3))

app = Flask(__name__)
app.secret_key = SECRET
app.config.update(SESSION_COOKIE_HTTPONLY=True, SESSION_COOKIE_SAMESITE="Lax",
                  PERMANENT_SESSION_LIFETIME=60 * 60 * 12)
LOCK = threading.Lock()


# ------------------------------------------------------------------ banco
def agora():
    return datetime.now(TZ).isoformat(timespec="seconds")


def conn():
    c = sqlite3.connect(DB, timeout=10)
    c.row_factory = sqlite3.Row
    return c


def init_db():
    with LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS alunos(
            id INTEGER PRIMARY KEY AUTOINCREMENT, turma TEXT NOT NULL, nome TEXT NOT NULL,
            sexo TEXT, idade INTEGER, UNIQUE(turma, nome));
        CREATE TABLE IF NOT EXISTS sessoes(
            id INTEGER PRIMARY KEY AUTOINCREMENT, turma TEXT NOT NULL, inicio TEXT NOT NULL,
            fim TEXT, status TEXT NOT NULL DEFAULT 'aberta');
        CREATE TABLE IF NOT EXISTS presencas(
            sessao_id INTEGER NOT NULL, aluno_id INTEGER NOT NULL,
            marcado_em TEXT, override INTEGER, final INTEGER,
            PRIMARY KEY(sessao_id, aluno_id));
        """)
        if c.execute("SELECT COUNT(*) FROM alunos").fetchone()[0] == 0 and os.path.exists(ALUNOS_CSV):
            with open(ALUNOS_CSV, encoding="utf-8") as f:
                for r in csv.DictReader(f):
                    c.execute("INSERT OR IGNORE INTO alunos(turma,nome,sexo,idade) VALUES(?,?,?,?)",
                              (r["turma"], r["nome"], r.get("sexo"), int(r["idade"]) if r.get("idade") else None))


PRESENTE_SQL = "COALESCE(p.final, CASE WHEN p.override IS NOT NULL THEN p.override WHEN p.marcado_em IS NOT NULL THEN 1 ELSE 0 END)"


def get_sessao(sid):
    with conn() as c:
        return c.execute("SELECT * FROM sessoes WHERE id=?", (sid,)).fetchone()


# ------------------------------------------------------------------ token do QR (rotativo)
def _tok(sid, janela):
    return hmac.new(SECRET.encode(), ("%s:%s" % (sid, janela)).encode(), hashlib.sha256).hexdigest()[:12]


def token_atual(sid):
    return _tok(sid, int(time.time() // ROTACAO))


def token_valido(sid, t):
    j = int(time.time() // ROTACAO)
    return bool(t) and any(hmac.compare_digest(t, _tok(sid, x)) for x in (j, j - 1))


# ------------------------------------------------------------------ autenticação do professor
_tentativas = {}


def prof_required(api=False):
    def deco(fn):
        @wraps(fn)
        def wrap(*a, **k):
            if not session.get("prof"):
                if api:
                    return jsonify(erro="não autenticado"), 401
                return redirect("/prof/login")
            return fn(*a, **k)
        return wrap
    return deco


# ------------------------------------------------------------------ HTML
HEAD = """<!DOCTYPE html><html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{{ titulo }}</title>
<script src="https://cdn.tailwindcss.com/3.4.17"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/qrcodejs/1.0.0/qrcode.min.js"></script>
</head><body class="bg-slate-50 text-slate-800 min-h-screen">"""


def pagina(titulo, corpo, **ctx):
    return render_template_string(HEAD + corpo + "</body></html>", titulo=titulo, **ctx)


HOME = """
<main class="max-w-md mx-auto p-6 space-y-5">
  <h1 class="text-2xl font-semibold mt-8">Chamada por QR Code</h1>
  <a href="/prof" class="block bg-indigo-600 text-white rounded-xl p-5 shadow">
    <p class="font-medium text-lg">Sou professor(a)</p><p class="text-sm opacity-90">Abrir e conferir a chamada da turma</p></a>
  <div class="bg-white border rounded-xl p-5">
    <p class="font-medium text-lg">Sou aluno(a)</p>
    <p class="text-sm text-slate-600">Abra a câmera do celular e aponte para o QR Code que está na tela da sala.</p></div>
</main>"""

LOGIN = """
<main class="max-w-sm mx-auto p-6 mt-16">
  <h1 class="text-xl font-semibold mb-4">Acesso do professor</h1>
  <form method="post" class="space-y-3 bg-white border rounded-xl p-5">
    <label class="text-sm" for="pin">PIN</label>
    <input id="pin" name="pin" type="password" inputmode="numeric" autofocus class="w-full border rounded-lg px-3 py-2">
    {% if erro %}<p class="text-sm text-rose-600">{{ erro }}</p>{% endif %}
    <button class="w-full bg-indigo-600 text-white rounded-lg py-2">Entrar</button>
  </form>
</main>"""

PAINEL = """
<main class="max-w-5xl mx-auto p-4 space-y-6">
  <header class="flex items-center justify-between"><h1 class="text-xl font-semibold">Painel do professor</h1>
    <a href="/prof/sair" class="text-sm text-slate-500 underline">Sair</a></header>
  <section><h2 class="font-medium mb-2">Abrir chamada</h2>
    <div id="turmas" class="grid grid-cols-2 md:grid-cols-5 gap-3"></div></section>
  <section class="bg-white border rounded-xl overflow-x-auto"><h2 class="font-medium p-4 pb-2">Últimas chamadas</h2>
    <table class="w-full text-sm"><thead class="text-left text-slate-500"><tr>
      <th class="px-4 py-2">Data</th><th>Turma</th><th>Situação</th><th>Presentes</th><th></th></tr></thead>
      <tbody id="hist"></tbody></table></section>
</main>
<script>
async function abrir(turma){
  const r = await fetch('/api/prof/abrir',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({turma})});
  const d = await r.json(); if(d.sid) location = '/prof/chamada/'+d.sid; else alert(d.erro||'Erro');
}
async function carregar(){
  const r = await fetch('/api/prof/painel'); if(r.status===401){location='/prof/login';return;}
  const d = await r.json();
  const box = document.getElementById('turmas'); box.innerHTML='';
  d.turmas.forEach(t=>{
    const b=document.createElement('button');
    b.className='bg-white border rounded-xl p-4 text-left hover:border-indigo-500 shadow-sm';
    const n=document.createElement('p'); n.className='font-medium'; n.textContent=t.turma;
    const s=document.createElement('p'); s.className='text-xs text-slate-500';
    s.textContent=t.alunos+' alunos'+(t.aberta?' · chamada aberta':'');
    if(t.aberta) b.classList.add('border-emerald-500');
    b.append(n,s); b.onclick=()=>abrir(t.turma); box.appendChild(b);
  });
  const tb=document.getElementById('hist'); tb.innerHTML='';
  d.historico.forEach(h=>{
    const tr=document.createElement('tr'); tr.className='border-t';
    const cel=(t)=>{const td=document.createElement('td'); td.className='px-4 py-2'; td.textContent=t; return td;};
    tr.append(cel(h.inicio.replace('T',' ').slice(0,16)),cel(h.turma),cel(h.status==='aberta'?'Aberta':'Finalizada'),cel(h.presentes+' / '+h.total));
    const td=document.createElement('td'); td.className='px-4 py-2 space-x-3';
    const a=document.createElement('a'); a.href='/prof/chamada/'+h.id; a.className='text-indigo-600 underline'; a.textContent='Abrir';
    const c=document.createElement('a'); c.href='/prof/sessao/'+h.id+'.csv'; c.className='text-slate-500 underline'; c.textContent='CSV';
    td.append(a,c); tr.appendChild(td); tb.appendChild(tr);
  });
  if(!d.historico.length){tb.innerHTML='<tr><td class="px-4 py-3 text-slate-500" colspan="5">Nenhuma chamada ainda.</td></tr>';}
}
carregar(); setInterval(carregar,5000);
</script>"""

CHAMADA = """
<main class="max-w-6xl mx-auto p-4 space-y-4">
  <header class="flex flex-wrap items-center justify-between gap-2">
    <div><a href="/prof" class="text-sm text-slate-500 underline">← Painel</a>
      <h1 class="text-xl font-semibold" id="titulo">Chamada</h1>
      <p class="text-sm text-slate-500" id="sub"></p></div>
    <div class="flex gap-2">
      <a id="btnCsv" href="/prof/sessao/{{ sid }}.csv" class="border rounded-lg px-3 py-2 text-sm bg-white">Baixar CSV</a>
      <button id="btnFim" class="bg-emerald-600 text-white rounded-lg px-4 py-2 text-sm font-medium">Finalizar chamada</button>
    </div>
  </header>
  <div class="grid lg:grid-cols-[320px_1fr] gap-4">
    <aside class="bg-white border rounded-xl p-4 text-center self-start" id="qrCard">
      <p class="text-sm font-medium mb-2">Alunos: aponte a câmera para o QR Code</p>
      <div id="qr" class="inline-block p-2 bg-white"></div>
      <p class="text-xs text-slate-500 mt-2">O QR se renova a cada <span id="rot"></span> s para valer só para quem está na sala.</p>
      <p class="text-xs text-slate-400" id="seg"></p>
      <button id="btnProj" class="mt-3 text-sm text-indigo-600 underline">Projetar QR em tela cheia</button>
    </aside>
    <section class="bg-white border rounded-xl">
      <div class="p-4 border-b space-y-2">
        <div class="flex flex-wrap gap-4 text-sm">
          <span>Presentes: <b id="nPres">0</b></span><span>Aguardando: <b id="nAg">0</b></span><span>Total: <b id="nTot">0</b></span>
        </div>
        <div class="h-2 bg-slate-100 rounded"><div id="barra" class="h-2 bg-emerald-500 rounded" style="width:0%"></div></div>
        <div class="flex gap-2 text-sm"><select id="filtro" class="border rounded-lg px-2 py-1 bg-white">
          <option value="todos">Todos</option><option value="marcou">Marcaram pelo celular</option><option value="falta">Ainda não marcaram</option></select></div>
      </div>
      <ul id="lista" class="divide-y"></ul>
    </section>
  </div>
  <p class="text-xs text-slate-500">Confira a lista: toque em <b>Presente/Falta</b> para corrigir qualquer nome (por exemplo, aluno sem celular ou alguém que marcou o nome do colega). Ao finalizar, a chamada é gravada e o QR deixa de valer.</p>
</main>
<div id="proj" class="hidden fixed inset-0 bg-white z-50 flex flex-col items-center justify-center cursor-pointer">
  <p class="text-3xl font-semibold mb-6" id="projTurma"></p><div id="qrBig"></div>
  <p class="mt-6 text-slate-500">Aponte a câmera do celular · toque para fechar</p></div>
<script>
const SID = {{ sid }};
const ROT = {{ rot }};
let estado = null, ultimaUrl = '';
const $ = s => document.querySelector(s);
$('#rot').textContent = ROT;

function desenharQR(el, url, tam){ el.innerHTML=''; new QRCode(el,{text:url,width:tam,height:tam,correctLevel:QRCode.CorrectLevel.M}); }

async function carregar(){
  const r = await fetch('/api/prof/estado/'+SID);
  if(r.status===401){location='/prof/login';return;}
  if(!r.ok){return;}
  estado = await r.json(); render(estado);
}
function render(d){
  const aberta = d.status==='aberta';
  $('#titulo').textContent = 'Chamada · '+d.turma;
  $('#sub').textContent = d.inicio.replace('T',' ').slice(0,16)+' · '+(aberta?'em andamento':'finalizada');
  $('#qrCard').classList.toggle('hidden',!aberta);
  $('#btnFim').classList.toggle('hidden',!aberta);
  if(aberta){
    const url = location.origin + d.qr_path;
    if(url!==ultimaUrl){ ultimaUrl=url; desenharQR($('#qr'),url,240); if(!$('#proj').classList.contains('hidden')) desenharQR($('#qrBig'),url,Math.min(420,innerWidth-60)); }
    $('#seg').textContent = 'Próxima troca em ~'+d.segundos+' s';
  }
  $('#nPres').textContent=d.presentes; $('#nTot').textContent=d.total; $('#nAg').textContent=d.total-d.presentes;
  $('#barra').style.width = (d.total? 100*d.presentes/d.total:0)+'%';
  const f = $('#filtro').value, ul = $('#lista'); ul.innerHTML='';
  d.alunos.filter(a=> f==='todos' || (f==='marcou'? a.aluno_marcou : !a.aluno_marcou)).forEach(a=>{
    const li=document.createElement('li'); li.className='flex items-center justify-between px-4 py-2.5';
    const esq=document.createElement('div');
    const nome=document.createElement('p'); nome.className='font-medium'; nome.textContent=a.nome;
    const info=document.createElement('p'); info.className='text-xs text-slate-500';
    let t = a.aluno_marcou ? 'Marcou no celular às '+a.hora : 'Ainda não marcou';
    if(a.override===1 && !a.aluno_marcou) t='Presença lançada pelo professor';
    if(a.override===0) t += ' · ajustado para falta pelo professor';
    info.textContent=t; esq.append(nome,info);
    const b=document.createElement('button'); b.disabled=!aberta;
    b.className='rounded-full px-4 py-1.5 text-sm font-medium '+(a.presente?'bg-emerald-100 text-emerald-800':'bg-rose-50 text-rose-700')+(aberta?'':' opacity-70');
    b.textContent=a.presente?'Presente':'Falta';
    b.onclick=()=>marcar(a.id,!a.presente);
    li.append(esq,b); ul.appendChild(li);
  });
}
async function marcar(id,presente){
  await fetch('/api/prof/marcar',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({sid:SID,aluno_id:id,presente})});
  carregar();
}
$('#filtro').onchange=()=>estado&&render(estado);
$('#btnFim').onclick=async()=>{
  if(!estado) return;
  if(!confirm('Finalizar a chamada de '+estado.turma+'?\\n'+estado.presentes+' presentes e '+(estado.total-estado.presentes)+' faltas. Depois disso o QR deixa de valer.')) return;
  await fetch('/api/prof/finalizar',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({sid:SID})});
  carregar();
};
$('#btnProj').onclick=()=>{ $('#projTurma').textContent=estado?estado.turma:''; $('#proj').classList.remove('hidden');
  desenharQR($('#qrBig'),ultimaUrl,Math.min(420,innerWidth-60)); };
$('#proj').onclick=()=>$('#proj').classList.add('hidden');
carregar(); setInterval(carregar,2000);
</script>"""

ALUNO = """
<main class="max-w-md mx-auto p-4 space-y-4">
  <header><p class="text-xs text-slate-500">Chamada</p><h1 class="text-xl font-semibold" id="turma"></h1></header>
  <div id="encerrada" class="hidden bg-amber-50 border border-amber-200 rounded-xl p-4 text-sm">Esta chamada foi encerrada pelo professor.</div>
  <div id="ok" class="hidden bg-emerald-50 border border-emerald-200 rounded-xl p-5 text-center space-y-2">
    <p class="text-3xl">✅</p><p class="font-medium">Presença registrada!</p>
    <p class="text-sm text-slate-600" id="meuNome"></p>
    <p class="text-xs text-slate-500">O professor vai conferir a lista. Se marcou o nome errado, desfaça e escolha de novo.</p>
    <button id="btnDesf" class="text-sm underline text-slate-600">Não sou eu · desfazer</button></div>
  <div id="area">
    <p class="text-sm text-slate-600 mb-2">Toque no <b>seu</b> nome:</p>
    <input id="busca" type="search" placeholder="Buscar meu nome" class="w-full border rounded-lg px-3 py-2 mb-3 bg-white">
    <ul id="lista" class="space-y-2"></ul></div>
  <div id="msg" class="hidden fixed bottom-4 left-4 right-4 bg-slate-900 text-white text-sm rounded-lg p-3 text-center"></div>
</main>
<div id="modal" class="hidden fixed inset-0 bg-black/40 flex items-end sm:items-center justify-center p-4 z-50">
  <div class="bg-white rounded-2xl p-5 w-full max-w-sm space-y-4">
    <p class="text-sm text-slate-500">Confirmar presença de</p><p class="text-xl font-semibold" id="mNome"></p>
    <div class="flex gap-2"><button id="mNao" class="flex-1 border rounded-lg py-2.5">Cancelar</button>
      <button id="mSim" class="flex-1 bg-indigo-600 text-white rounded-lg py-2.5 font-medium">Sou eu</button></div></div></div>
<script>
const SID = {{ sid }};
const $ = s => document.querySelector(s);
let alunos = [], meu = null, escolhido = null;
$('#turma').textContent = {{ turma|tojson }};
function aviso(t){ const m=$('#msg'); m.textContent=t; m.classList.remove('hidden'); setTimeout(()=>m.classList.add('hidden'),3500); }
async function carregar(){
  const r = await fetch('/api/aluno/lista/'+SID);
  if(!r.ok){ $('#area').classList.add('hidden'); $('#encerrada').classList.remove('hidden'); $('#encerrada').textContent='Link expirado. Escaneie novamente o QR Code da tela.'; return; }
  const d = await r.json();
  if(d.status!=='aberta'){ $('#area').classList.add('hidden'); $('#ok').classList.add('hidden'); $('#encerrada').classList.remove('hidden'); return; }
  alunos = d.alunos; meu = d.meu_id; desenhar();
}
function desenhar(){
  const jaMarquei = meu!==null;
  $('#ok').classList.toggle('hidden',!jaMarquei); $('#area').classList.toggle('hidden',jaMarquei);
  if(jaMarquei){ const a=alunos.find(x=>x.id===meu); $('#meuNome').textContent=a?a.nome:''; return; }
  const q = $('#busca').value.trim().toLowerCase(), ul=$('#lista'); ul.innerHTML='';
  alunos.filter(a=>!q||a.nome.toLowerCase().includes(q)).forEach(a=>{
    const li=document.createElement('li'); const b=document.createElement('button');
    b.className='w-full text-left rounded-xl border px-4 py-3 '+(a.marcado?'bg-slate-100 text-slate-400':'bg-white active:bg-indigo-50');
    b.disabled=a.marcado; b.textContent=a.nome+(a.marcado?'  ·  já registrado':'');
    b.onclick=()=>{ escolhido=a; $('#mNome').textContent=a.nome; $('#modal').classList.remove('hidden'); };
    li.appendChild(b); ul.appendChild(li);
  });
}
$('#busca').oninput=desenhar;
$('#mNao').onclick=()=>$('#modal').classList.add('hidden');
$('#mSim').onclick=async()=>{
  $('#modal').classList.add('hidden'); if(!escolhido) return;
  const r=await fetch('/api/aluno/marcar',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({sid:SID,aluno_id:escolhido.id})});
  const d=await r.json(); if(!r.ok) aviso(d.erro||'Não foi possível marcar.'); carregar();
};
$('#btnDesf').onclick=async()=>{ await fetch('/api/aluno/desfazer',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({sid:SID})}); carregar(); };
carregar(); setInterval(carregar,4000);
</script>"""

MSG = """<main class="max-w-md mx-auto p-6 mt-16 text-center space-y-3"><p class="text-4xl">{{ icone }}</p>
<h1 class="text-xl font-semibold">{{ cab }}</h1><p class="text-slate-600">{{ texto }}</p></main>"""


# ------------------------------------------------------------------ páginas
@app.get("/")
def home():
    return pagina("Chamada", HOME)


@app.route("/prof/login", methods=["GET", "POST"])
def prof_login():
    erro = None
    if request.method == "POST":
        ip = request.headers.get("CF-Connecting-IP") or request.remote_addr
        agora_t = time.time()
        recentes = [t for t in _tentativas.get(ip, []) if agora_t - t < 600]
        if len(recentes) >= 8:
            erro = "Muitas tentativas. Aguarde alguns minutos."
        elif hmac.compare_digest(request.form.get("pin", ""), PIN):
            session.permanent = True
            session["prof"] = True
            return redirect("/prof")
        else:
            recentes.append(agora_t)
            erro = "PIN incorreto."
        _tentativas[ip] = recentes
    return pagina("Acesso do professor", LOGIN, erro=erro)


@app.get("/prof/sair")
def prof_sair():
    session.pop("prof", None)
    return redirect("/")


@app.get("/prof")
@prof_required()
def prof_painel():
    return pagina("Painel do professor", PAINEL)


@app.get("/prof/chamada/<int:sid>")
@prof_required()
def prof_chamada(sid):
    if not get_sessao(sid):
        return pagina("Chamada", MSG, icone="❓", cab="Chamada não encontrada", texto="Volte ao painel."), 404
    return pagina("Chamada", CHAMADA, sid=sid, rot=ROTACAO)


@app.get("/e/<int:sid>")
def aluno_pagina(sid):
    s = get_sessao(sid)
    if not s or s["status"] != "aberta":
        return pagina("Chamada encerrada", MSG, icone="🔒", cab="Chamada encerrada",
                      texto="O professor já finalizou esta chamada."), 410
    liberados = session.get("ok", [])
    t = request.args.get("t")
    if t:
        if token_valido(sid, t):
            if sid not in liberados:
                liberados = (liberados + [sid])[-5:]
            session["ok"] = liberados
            session.permanent = True
            return redirect("/e/%d" % sid)  # tira o token da barra de endereço
        if sid not in liberados:
            return pagina("QR expirado", MSG, icone="⏱️", cab="QR Code expirado",
                          texto="Aponte a câmera novamente para o QR Code que está na tela da sala."), 403
    if sid not in liberados:
        return pagina("Escaneie o QR", MSG, icone="📷", cab="Escaneie o QR Code",
                      texto="Use a câmera do celular no QR Code da tela da sala para entrar na chamada."), 403
    return pagina("Marcar presença", ALUNO, sid=sid, turma=s["turma"])


# ------------------------------------------------------------------ API do aluno
def _aluno_liberado(sid):
    return sid in session.get("ok", [])


@app.get("/api/aluno/lista/<int:sid>")
def api_aluno_lista(sid):
    s = get_sessao(sid)
    if not s or not _aluno_liberado(sid):
        return jsonify(erro="sem acesso"), 403
    with conn() as c:
        rows = c.execute("""SELECT a.id, a.nome, (p.marcado_em IS NOT NULL OR p.override IS NOT NULL) AS marcado
                            FROM alunos a LEFT JOIN presencas p ON p.aluno_id=a.id AND p.sessao_id=?
                            WHERE a.turma=? ORDER BY a.nome""", (sid, s["turma"])).fetchall()
    meu = session.get("marcou", {}).get(str(sid))
    return jsonify(status=s["status"], turma=s["turma"], meu_id=meu,
                   alunos=[{"id": r["id"], "nome": r["nome"], "marcado": bool(r["marcado"]) and r["id"] != meu}
                           for r in rows])


@app.post("/api/aluno/marcar")
def api_aluno_marcar():
    d = request.get_json(silent=True) or {}
    sid, aid = d.get("sid"), d.get("aluno_id")
    if not isinstance(sid, int) or not isinstance(aid, int):
        return jsonify(erro="requisição inválida"), 400
    s = get_sessao(sid)
    if not s or not _aluno_liberado(sid):
        return jsonify(erro="Escaneie o QR Code novamente."), 403
    if s["status"] != "aberta":
        return jsonify(erro="Chamada encerrada."), 409
    marcou = session.get("marcou", {})
    if str(sid) in marcou:
        return jsonify(erro="Este celular já registrou uma presença. Desfaça antes de marcar outro nome."), 409
    with LOCK, conn() as c:
        ok = c.execute("SELECT 1 FROM alunos WHERE id=? AND turma=?", (aid, s["turma"])).fetchone()
        if not ok:
            return jsonify(erro="Aluno não pertence a esta turma."), 400
        cur = c.execute("""UPDATE presencas SET marcado_em=? WHERE sessao_id=? AND aluno_id=?
                           AND marcado_em IS NULL AND override IS NULL""", (agora(), sid, aid))
        if cur.rowcount == 0:
            return jsonify(erro="Esse nome já foi registrado. Fale com o professor."), 409
    marcou[str(sid)] = aid
    session["marcou"] = dict(list(marcou.items())[-5:])
    return jsonify(ok=True)


@app.post("/api/aluno/desfazer")
def api_aluno_desfazer():
    d = request.get_json(silent=True) or {}
    sid = d.get("sid")
    s = get_sessao(sid) if isinstance(sid, int) else None
    marcou = session.get("marcou", {})
    aid = marcou.get(str(sid))
    if not s or s["status"] != "aberta" or aid is None:
        return jsonify(erro="nada a desfazer"), 409
    with LOCK, conn() as c:
        c.execute("UPDATE presencas SET marcado_em=NULL WHERE sessao_id=? AND aluno_id=?", (sid, aid))
    marcou.pop(str(sid), None)
    session["marcou"] = marcou
    return jsonify(ok=True)


# ------------------------------------------------------------------ API do professor
@app.get("/api/prof/painel")
@prof_required(api=True)
def api_prof_painel():
    with conn() as c:
        turmas = []
        for t in TURMAS:
            n = c.execute("SELECT COUNT(*) FROM alunos WHERE turma=?", (t,)).fetchone()[0]
            ab = c.execute("SELECT 1 FROM sessoes WHERE turma=? AND status='aberta'", (t,)).fetchone()
            turmas.append({"turma": t, "alunos": n, "aberta": bool(ab)})
        hist = c.execute("""SELECT s.id, s.turma, s.inicio, s.status,
                 (SELECT COUNT(*) FROM presencas p WHERE p.sessao_id=s.id) AS total,
                 (SELECT COALESCE(SUM(%s),0) FROM presencas p WHERE p.sessao_id=s.id) AS presentes
                 FROM sessoes s ORDER BY s.id DESC LIMIT 20""" % PRESENTE_SQL).fetchall()
    return jsonify(turmas=turmas, historico=[dict(h) for h in hist])


@app.post("/api/prof/abrir")
@prof_required(api=True)
def api_prof_abrir():
    turma = (request.get_json(silent=True) or {}).get("turma")
    if turma not in TURMAS:
        return jsonify(erro="turma inválida"), 400
    with LOCK, conn() as c:
        ex = c.execute("SELECT id FROM sessoes WHERE turma=? AND status='aberta'", (turma,)).fetchone()
        if ex:
            return jsonify(sid=ex["id"])
        if c.execute("SELECT COUNT(*) FROM alunos WHERE turma=?", (turma,)).fetchone()[0] == 0:
            return jsonify(erro="Turma sem alunos cadastrados."), 400
        sid = c.execute("INSERT INTO sessoes(turma,inicio) VALUES(?,?)", (turma, agora())).lastrowid
        c.execute("INSERT INTO presencas(sessao_id,aluno_id) SELECT ?, id FROM alunos WHERE turma=?", (sid, turma))
    return jsonify(sid=sid)


def _estado(sid):
    s = get_sessao(sid)
    if not s:
        return None
    with conn() as c:
        rows = c.execute("""SELECT a.id, a.nome, a.idade, p.marcado_em, p.override, p.final
                            FROM presencas p JOIN alunos a ON a.id=p.aluno_id
                            WHERE p.sessao_id=? ORDER BY a.nome""", (sid,)).fetchall()
    alunos = []
    for r in rows:
        if r["final"] is not None:
            pres = bool(r["final"])
        elif r["override"] is not None:
            pres = bool(r["override"])
        else:
            pres = r["marcado_em"] is not None
        alunos.append({"id": r["id"], "nome": r["nome"], "idade": r["idade"],
                       "aluno_marcou": r["marcado_em"] is not None,
                       "hora": r["marcado_em"][11:16] if r["marcado_em"] else None,
                       "override": r["override"], "presente": pres})
    out = {"sid": sid, "turma": s["turma"], "inicio": s["inicio"], "fim": s["fim"], "status": s["status"],
           "alunos": alunos, "total": len(alunos), "presentes": sum(a["presente"] for a in alunos)}
    if s["status"] == "aberta":
        out["qr_path"] = "/e/%d?t=%s" % (sid, token_atual(sid))
        out["segundos"] = ROTACAO - int(time.time() % ROTACAO)
    return out


@app.get("/api/prof/estado/<int:sid>")
@prof_required(api=True)
def api_prof_estado(sid):
    e = _estado(sid)
    return (jsonify(e), 200) if e else (jsonify(erro="não encontrada"), 404)


@app.post("/api/prof/marcar")
@prof_required(api=True)
def api_prof_marcar():
    d = request.get_json(silent=True) or {}
    sid, aid, pres = d.get("sid"), d.get("aluno_id"), d.get("presente")
    s = get_sessao(sid) if isinstance(sid, int) else None
    if not s or s["status"] != "aberta" or not isinstance(aid, int):
        return jsonify(erro="chamada indisponível"), 409
    val = None if pres is None else int(bool(pres))
    with LOCK, conn() as c:
        c.execute("UPDATE presencas SET override=? WHERE sessao_id=? AND aluno_id=?", (val, sid, aid))
    return jsonify(ok=True)


@app.post("/api/prof/finalizar")
@prof_required(api=True)
def api_prof_finalizar():
    sid = (request.get_json(silent=True) or {}).get("sid")
    s = get_sessao(sid) if isinstance(sid, int) else None
    if not s:
        return jsonify(erro="não encontrada"), 404
    if s["status"] == "aberta":
        with LOCK, conn() as c:
            c.execute("""UPDATE presencas SET final=CASE WHEN override IS NOT NULL THEN override
                         WHEN marcado_em IS NOT NULL THEN 1 ELSE 0 END WHERE sessao_id=?""", (sid,))
            c.execute("UPDATE sessoes SET status='finalizada', fim=? WHERE id=?", (agora(), sid))
    e = _estado(sid)
    return jsonify(presentes=e["presentes"], total=e["total"])


# ------------------------------------------------------------------ exportação
def exportar_csv_texto(sid=None):
    """CSV (UTF-8 com BOM, abre direto no Excel). sid=None exporta todas as chamadas."""
    with conn() as c:
        q = """SELECT s.inicio, s.turma, s.status, a.nome, a.idade, p.marcado_em, p.override,
                      p.final, ( %s ) AS presente
               FROM presencas p JOIN sessoes s ON s.id=p.sessao_id JOIN alunos a ON a.id=p.aluno_id""" % PRESENTE_SQL
        rows = c.execute(q + (" WHERE s.id=?" if sid else "") + " ORDER BY s.id, a.nome",
                         (sid,) if sid else ()).fetchall()
    buf = io.StringIO()
    w = csv.writer(buf, delimiter=";")
    w.writerow(["data", "hora_inicio", "turma", "situacao_chamada", "aluno", "idade", "status", "origem", "hora_marcacao"])
    for r in rows:
        if r["override"] is not None and (r["marcado_em"] is None or r["override"] == 0):
            origem = "Professor"
        elif r["marcado_em"]:
            origem = "Aluno (celular)"
        else:
            origem = "—"
        w.writerow([r["inicio"][:10], r["inicio"][11:16], r["turma"], r["status"], r["nome"], r["idade"],
                    "Presente" if r["presente"] else "Falta", origem,
                    r["marcado_em"][11:16] if r["marcado_em"] else ""])
    return "\ufeff" + buf.getvalue()


@app.get("/prof/sessao/<int:sid>.csv")
@prof_required()
def prof_csv(sid):
    s = get_sessao(sid)
    if not s:
        return "não encontrada", 404
    return Response(exportar_csv_texto(sid), mimetype="text/csv; charset=utf-8",
                    headers={"Content-Disposition": "attachment; filename=chamada_%d.csv" % sid})


@app.get("/prof/exportar.csv")
@prof_required()
def prof_csv_todas():
    return Response(exportar_csv_texto(), mimetype="text/csv; charset=utf-8",
                    headers={"Content-Disposition": "attachment; filename=chamadas.csv"})


init_db()

if __name__ == "__main__":
    print("PIN do professor:", PIN)
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 5000)))

Writing app_chamada.py


## 4. Gerar os alunos a partir da planilha

In [4]:
from google.colab import files
import os
print("Envie a planilha .xlsx (a mesma do dashboard):")
up = files.upload()
PLANILHA = next(iter(up))

import gerar_alunos
alunos = gerar_alunos.gerar(PLANILHA, "alunos_ficticios.csv", por_turma=24, seed=7)
alunos.groupby("turma").agg(alunos=("nome", "size"), idade_media=("idade", "mean"),
                            meninos=("sexo", lambda s: (s == "Masculino").sum())).round(1)

Envie a planilha .xlsx (a mesma do dashboard):


Saving Inscritos - Andamento v2.xlsx to Inscritos - Andamento v2.xlsx


,alunos,idade_media,meninos
turma,,,
Artes,24,9.8,8
Balé,24,9.7,4
Futebol,24,9.9,20
Judô,24,9.5,16
Teatro,24,9.5,10


## 5. Subir o servidor e gerar o link público

* Troque o **PIN** abaixo antes de rodar.
* O link `trycloudflare.com` é **público**: qualquer pessoa com ele chega à tela inicial, mas `/prof` exige PIN e o link do aluno só funciona com o QR da tela.
* **O professor deve abrir o link no próprio navegador** (não em `localhost`), senão o QR sai com o endereço errado.

In [5]:
import os, re, subprocess, threading, time, urllib.request, stat

os.environ["CHAMADA_PIN"] = "1234"        # <<< TROQUE
os.environ["CHAMADA_DB"] = "chamada.db"
os.environ["CHAMADA_ROTACAO"] = "30"      # segundos de vida do QR

import importlib, app_chamada
importlib.reload(app_chamada)
threading.Thread(target=lambda: app_chamada.app.run(host="127.0.0.1", port=5000, use_reloader=False, threaded=True),
                 daemon=True).start()
time.sleep(2)

# túnel gratuito (sem conta) da Cloudflare
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", os.stat("cloudflared").st_mode | stat.S_IEXEC)
proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for linha in proc.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", linha)
    if m:
        URL = m.group(0); break
print("\n==> Professor, abra este link:", URL, "\n    PIN:", os.environ["CHAMADA_PIN"])
threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()   # mantém o túnel drenando o log

 * Serving Flask app 'app_chamada'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



==> Professor, abra este link: https://permit-september-trinity-funeral.trycloudflare.com 
    PIN: 1234


### Como testar sozinho
1. Abra o link no computador → *Sou professor* → PIN → clique em **Balé** → aparece o QR.
2. No celular, aponte a câmera para o QR (ou use o botão *Projetar QR em tela cheia*).
3. Escolha um nome e confirme. No computador a linha muda para **Presente** em até 2 s.
4. Corrija o que for preciso e clique em **Finalizar chamada**.

**Atenção:** mantenha esta aba do Colab aberta durante a aula; se a sessão do Colab desligar, o link muda e os dados locais se perdem.
Por isso, baixe o histórico na célula 6 ao fim de cada aula.

## 6. Baixar o histórico de chamadas (CSV)

In [6]:
texto = app_chamada.exportar_csv_texto()
with open("chamadas.csv", "w", encoding="utf-8") as f:
    f.write(texto)
files.download("chamadas.csv")
print(texto.count("\n") - 1, "linhas exportadas")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

0 linhas exportadas


## 7. Do protótipo ao uso real

| Tema | O que fazer |
|---|---|
| **Hospedagem** | Os mesmos arquivos rodam em Render, Railway, Fly.io ou servidor da escola: `pip install flask gunicorn` e `gunicorn -w 1 app_chamada:app` (use **1** worker por causa do SQLite e do QR). |
| **Segredos** | Defina `CHAMADA_PIN` e `CHAMADA_SECRET` como variáveis de ambiente; o PIN padrão `1234` é só para teste. |
| **Alunos reais** | Substitua `alunos_ficticios.csv` por um CSV com `turma,nome,sexo,idade` (apague `chamada.db` para recarregar). Como são crianças e adolescentes, mantenha nomes abreviados e só a lista da turma visível. |
| **Fraude (colega marcando pelo outro)** | O QR rotativo, o limite de **um nome por celular** e a conferência do professor reduzem o risco, mas não eliminam. Para mais rigor, o professor pode chamar apenas quem está ausente da tela. |
| **LGPD** | Dados de menores: acesso restrito ao professor, base protegida e política de retenção definida. |